# Figure creation with new atlases (human HCP-MMP1 + macaque MacBNA)

This notebook is the runnable companion to the
[Figure creation](../docs/tutorials/figure_creation.md) tutorial. It extracts
**real** ROI coordinates/names from two atlas volumes, fabricates **synthetic**
networks (fixed random seeds), and renders every figure used on the docs page.

> All connectivity matrices, modules, node sizes and metrics here are synthetic —
> only the coordinates and ROI names are real. The atlas volumes/meshes are large
> external data you supply under `test_files/tutorial_files/parcellation and meshes/`.

In [ ]:
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as IPyImage

# Walk up to the repo root so relative paths resolve from anywhere.
here = Path.cwd()
repo_root = here
for p in [here, *here.parents]:
    if (p / 'test_files').is_dir() and (p / 'tutorial').is_dir():
        repo_root = p
        break
print('Repo root:', repo_root)

TF = repo_root / 'test_files' / 'tutorial_files'
DEMO = TF / 'new_atlas_demo'
PARC = TF / 'parcellation and meshes'
IMG = repo_root / 'docs' / 'images' / 'figure_creation'
IMG.mkdir(parents=True, exist_ok=True)
HTML_TMP = Path(tempfile.mkdtemp(prefix='figcreate_'))   # throwaway HTML dummies
sys.path.append(str(DEMO))

from HarrisLabPlotting import (
    load_mesh_file,
    create_brain_connectivity_plot,
    create_brain_connectivity_plot_with_modularity,
)

HUMAN_MESH = PARC / 'HCPMMP1_on_MNI152_ICBM2009a_nlin_hd_0.obj'
MONKEY_MESH = PARC / 'monkey_brain_mesh_MacBNA.obj'
EXAMPLE_EDGE = TF / 'node_edge_28' / 'connectivity_28.edge'
print('Inputs present:',
      HUMAN_MESH.exists(), MONKEY_MESH.exists(), EXAMPLE_EDGE.exists())

## 0. Generate the data

`generate_figure_data.py` builds both LUTs (from each atlas's own label table),
extracts real ROI coordinates, and fabricates the synthetic networks — all
deterministically. It also runs an alignment sanity-check at the end.

In [ ]:
import generate_figure_data as gfd
gfd.main()

## 1. Human — modularity across six views (2x3 grid)

A clean 2x3 multi-view grid of a 50-edge / 5-module synthetic network: nodes
colored by module (the default), no title and no edge-width key, module legend in
the first panel only.

In [ ]:
hv, hf = load_mesh_file(str(HUMAN_MESH))
coords = pd.read_csv(DEMO / 'human' / 'hcpmmp1_coords.csv')

out = IMG / 'human_modularity_grid_2x3.png'
create_brain_connectivity_plot_with_modularity(
    vertices=hv, faces=hf, roi_coords_df=coords,
    connectivity_matrix=str(DEMO / 'human' / 'hcpmmp1_modular_network.csv'),
    module_assignments=str(DEMO / 'human' / 'hcpmmp1_modules.csv'),
    multi_view=['anterior', 'posterior', 'left', 'right', 'superior', 'oblique'],
    multi_view_grid=(2, 3), multi_view_panel_size=(700, 700),
    show_node_labels=False, show_width_legend=False, plot_title='',
    zoom=1.3, image_dpi=150,
    save_path=str(HTML_TMP / 'human_grid.html'), export_image=str(out),
)
IPyImage(filename=str(out))

## 2. Monkey — per-node sizes and legend keys

Reuse the bundled 28-node example topology on 28 real MacBNA ROIs. Node sizes are
pre-scaled from a synthetic participation coefficient.

In [ ]:
mv, mf = load_mesh_file(str(MONKEY_MESH))
mcoords = pd.read_csv(DEMO / 'monkey' / 'coords_28.csv')
sizes = str(DEMO / 'monkey' / 'sizes_from_pc.csv')
metrics = str(DEMO / 'monkey' / 'metrics.csv')

common = dict(
    vertices=mv, faces=mf, roi_coords_df=mcoords,
    connectivity_matrix=str(EXAMPLE_EDGE), camera_view='oblique',
    show_node_labels=False, image_dpi=150, zoom=1.5,
)

# (a) vector sizes + scaled edges -> both keys appear automatically
out = IMG / 'monkey_size_key.png'
create_brain_connectivity_plot(
    node_size=sizes, edge_width=(1.0, 8.0), node_size_scale=0.5,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

In [ ]:
# (b) scalar size + fixed width -> both keys auto-skipped
out = IMG / 'monkey_no_keys.png'
create_brain_connectivity_plot(
    node_size=10, edge_width=2.0,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

In [ ]:
# (c) metric-labeled size key (participation coefficient)
out = IMG / 'monkey_metric_key.png'
create_brain_connectivity_plot(
    node_size=sizes, node_metrics=metrics,
    node_size_legend_metric='participation_coef',
    edge_width=(1.0, 8.0), node_size_scale=0.5,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

## 3. Monkey — default vs. customized

The same network rendered two ways: default (scalar size, fixed width, purple
nodes) vs. customized (PC-scaled sizes, module colors + border, scaled edges,
metric-labeled key).

In [ ]:
modules = (np.arange(len(mcoords)) % 4) + 1  # synthetic 4-module coloring

create_brain_connectivity_plot(
    node_size=8, edge_width=2.0,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(IMG / 'monkey_default.png'), **common)

create_brain_connectivity_plot(
    node_size=sizes, node_size_scale=0.5,
    node_color=modules, node_border_color='black',
    node_metrics=metrics, node_size_legend_metric='participation_coef',
    edge_width=(1.0, 8.0),
    save_path=str(HTML_TMP / 'm.html'), export_image=str(IMG / 'monkey_customized.png'), **common)

IPyImage(filename=str(IMG / 'monkey_customized.png'))

---

**Tip:** before plotting any new atlas, run the pre-flight checks in
[Checking atlas/mesh alignment](../docs/how_to/check_atlas_mesh_alignment.md), e.g.

```bash
hlplot utils check-alignment \
  --coords test_files/tutorial_files/new_atlas_demo/human/hcpmmp1_coords.csv \
  --mesh "test_files/tutorial_files/parcellation and meshes/HCPMMP1_on_MNI152_ICBM2009a_nlin_hd_0.obj"
```